# Raw Signal Comparison: 05-02-2026 vs 25-03-2026
Before realignment vs after realignment — Analog and Photon Counting curtain plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

PROTO_BASE = Path("/Users/thunthita/LidarNRBPipeline/LIDar/src/OutputPictCSV")
OUT_DIR    = Path("/Users/thunthita/LidarNRBPipeline/LIDar/ExampleCode/ColorPlot/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DAY_BEFORE = "05-02-2026"   # before realignment
DAY_AFTER  = "25-03-2026"   # after realignment

In [ ]:
def load_raw_day(day: str, col: str) -> pd.DataFrame:
    """Load one raw column (e.g. 'analog' or 'photon_counting') for all
    timestamps of a day into a (range_m × timestamp) DataFrame."""
    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    csvs = sorted(PROTO_BASE.glob(f"{day}-????/{day}-????_processed.csv"))
    if not csvs:
        raise FileNotFoundError(f"No processed CSVs found for {day}")

    cols = {}
    for csv in csvs:
        hhmm = csv.parent.name[-4:]   # e.g. '0005'
        ts = date_obj.replace(hour=int(hhmm[:2]), minute=int(hhmm[2:]), second=0)
        df = pd.read_csv(csv)
        if {"range_m", col}.issubset(df.columns):
            cols[ts] = pd.to_numeric(
                df.set_index("range_m")[col], errors="coerce"
            )

    Z = pd.DataFrame(cols).sort_index().sort_index(axis=1)
    print(f"  {day} | {col}: {Z.shape}  "
          f"range {Z.index.min():.0f}–{Z.index.max():.0f} m  "
          f"| {Z.columns[0].strftime('%H:%M')} → {Z.columns[-1].strftime('%H:%M')}")
    return Z


print("Loading analog...")
Z_analog_before = load_raw_day(DAY_BEFORE, "analog")
Z_analog_after  = load_raw_day(DAY_AFTER,  "analog")

print("Loading photon_counting...")
Z_pc_before = load_raw_day(DAY_BEFORE, "photon_counting")
Z_pc_after  = load_raw_day(DAY_AFTER,  "photon_counting")

In [ ]:
def plot_raw_comparison(
    Z_before, Z_after,
    label: str,
    day_before: str,
    day_after: str,
    clim=None,
    scale="linear",
    ylim=(0, 15000),
    cmap="jet",
    figsize=(14, 8),
    save=True,
):
    """Side-by-side curtain plots: before (left) vs after (right) realignment."""

    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=False)

    for ax, Z, day, title in [
        (axes[0], Z_before, day_before, f"{day_before}  (before realignment)"),
        (axes[1], Z_after,  day_after,  f"{day_after}   (after realignment)"),
    ]:
        times = Z.columns
        ranges = Z.index.to_numpy()

        Zplot = Z.to_numpy(dtype=float)
        if scale == "log":
            Zplot = np.log10(np.where(Zplot > 0, Zplot, np.nan))

        vmin, vmax = (clim if clim else (np.nanpercentile(Zplot, 1),
                                         np.nanpercentile(Zplot, 99)))

        im = ax.pcolormesh(
            mdates.date2num(times), ranges, Zplot,
            cmap=cmap, vmin=vmin, vmax=vmax, shading="nearest"
        )
        ax.set_ylim(*ylim)
        ax.set_ylabel("Range (m)")
        ax.set_title(title, fontsize=11)
        ax.xaxis_date()
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        plt.colorbar(im, ax=ax, label=label + (" [log₁₀]" if scale == "log" else ""))

    fig.suptitle(
        f"Raw {label} — Before vs After Realignment\n"
        f"({day_before} vs {day_after})",
        fontsize=13, y=1.01
    )
    fig.tight_layout()

    if save:
        fname = OUT_DIR / f"raw_{label.replace(' ','_')}_{scale}_{day_before}_vs_{day_after}.png"
        fig.savefig(fname, dpi=150, bbox_inches="tight")
        print(f"Saved: {fname.name}")

    plt.show()
    plt.close(fig)

In [ ]:
# ── Analog — linear scale ─────────────────────────────────────────────────────
plot_raw_comparison(
    Z_analog_before, Z_analog_after,
    label="Analog (mV)",
    day_before=DAY_BEFORE, day_after=DAY_AFTER,
    scale="linear",
    clim=(0, 50),
    ylim=(0, 15000),
)

In [ ]:
# ── Analog — log scale ────────────────────────────────────────────────────────
plot_raw_comparison(
    Z_analog_before, Z_analog_after,
    label="Analog (mV)",
    day_before=DAY_BEFORE, day_after=DAY_AFTER,
    scale="log",
    clim=(0, 2),
    ylim=(0, 15000),
)

In [ ]:
# ── Photon Counting — linear scale ────────────────────────────────────────────
plot_raw_comparison(
    Z_pc_before, Z_pc_after,
    label="Photon Counting (counts/bin)",
    day_before=DAY_BEFORE, day_after=DAY_AFTER,
    scale="linear",
    clim=(0, 100),
    ylim=(0, 15000),
)

In [ ]:
# ── Photon Counting — log scale ───────────────────────────────────────────────
plot_raw_comparison(
    Z_pc_before, Z_pc_after,
    label="Photon Counting (counts/bin)",
    day_before=DAY_BEFORE, day_after=DAY_AFTER,
    scale="log",
    clim=(0, 2),
    ylim=(0, 15000),
)

In [ ]:
# ── Single-timestamp profile overlay: noon comparison ─────────────────────────
# Pick one timestamp that exists in both days (e.g. 12:05)
COMPARE_HHMM = "1205"

def get_profile(Z: pd.DataFrame, hhmm: str):
    """Return range_m, values for the timestamp matching hhmm."""
    match = [c for c in Z.columns if c.strftime("%H%M") == hhmm]
    if not match:
        return None, None
    return Z.index.to_numpy(), Z[match[0]].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (Z_b, Z_a, label) in zip(
    axes,
    [
        (Z_analog_before,  Z_analog_after,  "Analog (mV)"),
        (Z_pc_before,      Z_pc_after,      "Photon Counting (counts/bin)"),
    ]
):
    r_b, v_b = get_profile(Z_b, COMPARE_HHMM)
    r_a, v_a = get_profile(Z_a, COMPARE_HHMM)

    if r_b is not None:
        ax.plot(v_b, r_b, label=f"{DAY_BEFORE} (before)", linewidth=1.5)
    if r_a is not None:
        ax.plot(v_a, r_a, label=f"{DAY_AFTER} (after)",  linewidth=1.5, linestyle="--")

    ax.set_xlabel(label)
    ax.set_ylabel("Range (m)")
    ax.set_ylim(0, 15000)
    ax.set_title(f"{label}\n@ {COMPARE_HHMM[:2]}:{COMPARE_HHMM[2:]}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Profile at {COMPARE_HHMM[:2]}:{COMPARE_HHMM[2:]} — Before vs After Realignment",
    fontsize=12
)
fig.tight_layout()

fname = OUT_DIR / f"profile_{COMPARE_HHMM}_{DAY_BEFORE}_vs_{DAY_AFTER}.png"
fig.savefig(fname, dpi=150, bbox_inches="tight")
print(f"Saved: {fname.name}")
plt.show()
plt.close(fig)